# Shipment Delay Prediction System
### End-to-End Machine Learning Portfolio Project

---

## 1. Introduction

### Problem Definition
In modern logistics, shipment delays are one of the most costly operational challenges. Late deliveries erode customer trust, increase operational costs, and disrupt supply chains. This project builds an **intelligent Shipment Delay Prediction System** using real-world delivery logistics data.

### Dataset Schema (25,000 rows × 15 columns)
| Column | Type | Description |
|---|---|---|
| `delivery_id` | int | Unique shipment identifier |
| `delivery_partner` | str | Courier company (Delhivery, XpressBees, etc.) |
| `package_type` | str | Category of goods |
| `vehicle_type` | str | Transport vehicle |
| `delivery_mode` | str | Same day / Express / Two day |
| `region` | str | Geographic zone |
| `weather_condition` | str | Weather at time of dispatch |
| `distance_km` | float | Route distance |
| `package_weight_kg` | float | Package weight |
| `delivery_time_hours` | float | **Actual** delivery time |
| `expected_time_hours` | float | **Promised** delivery time |
| `delayed` | str | Ground-truth delay flag (yes/no) |
| `delivery_status` | str | Final status |
| `delivery_rating` | int | Customer rating (1–5) |
| `delivery_cost` | float | Shipment cost |

### Two ML Targets
- **Classification** → `delay_flag` (0/1): Will this shipment be late?
- **Regression** → `delay_duration` (hours): By how many hours?

### Business Value
- Proactively alert operations teams before a delay becomes a customer complaint  
- Identify which routes, partners, and conditions drive the most delays  
- Prioritise re-routing resources on high-risk shipments

---
## 2. Environment Setup & Imports

In [ ]:
import warnings, os, json, datetime
warnings.filterwarnings('ignore')

# ── Data ────────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Preprocessing & Pipeline ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# ── Classification Models ─────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# ── Regression Models ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ── Optional: XGBoost / LightGBM ─────────────────────────────────────────────
try:
    import xgboost as xgb
    XGB_AVAILABLE = True
except ImportError:
    XGB_AVAILABLE = False

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except ImportError:
    LGB_AVAILABLE = False

# ── Explainability ────────────────────────────────────────────────────────────
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

# ── Global Settings ───────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ── Extra Models ─────────────────────────────────────────────────────────────
from sklearn.ensemble import ExtraTreesClassifier, ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.svm import SVC
from sklearn.linear_model import Lasso, ElasticNet

try:
    from catboost import CatBoostClassifier, CatBoostRegressor
    CB_AVAILABLE = True
except ImportError:
    CB_AVAILABLE = False

print(f"CatBoost: {'✅' if CB_AVAILABLE else '❌  pip install catboost'}")

print(f"NumPy  {np.__version__} | Pandas {pd.__version__}")
print(f"XGBoost : {'✅' if XGB_AVAILABLE  else '❌  pip install xgboost'}")
print(f"LightGBM: {'✅' if LGB_AVAILABLE  else '❌  pip install lightgbm'}")
print(f"SHAP    : {'✅' if SHAP_AVAILABLE else '❌  pip install shap'}")

---
## 3. Data Loading

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Robust Data Loading & Fixing Time Columns
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np

DATA_PATH = '/kaggle/input/datasets/ayeshaseherr/delivery-logistics-dataset/Delivery_Logistics.csv'

DTYPE_MAP = {
    'delivery_id'          : 'id',
    'delivery_partner'     : 'cat',
    'package_type'         : 'cat',
    'vehicle_type'         : 'cat',
    'delivery_mode'        : 'cat',
    'region'               : 'cat',
    'weather_condition'    : 'cat',
    'distance_km'          : 'num',
    'package_weight_kg'    : 'num',
    'delivery_time_hours'  : 'num',
    'expected_time_hours'  : 'num',
    'delayed'              : 'label',
    'delivery_status'      : 'cat',
    'delivery_rating'      : 'num',
    'delivery_cost'        : 'num',
}

# Load as string
df_raw = pd.read_csv(DATA_PATH, dtype=str)
print('Raw shape:', df_raw.shape)

# ─────────────────────────────────────────────────────────────────────────────
# Time columns are stored as nanosecond-epoch timestamps:
#   '1970-01-01 00:00:00.000000008'  → the nanosecond suffix IS the hours value
#   e.g. .000000008  → 8 hours,   .000000016 → 16 hours
# We extract the integer after the last '.' directly.
# ─────────────────────────────────────────────────────────────────────────────

def extract_hours(x):
    try:
        s = str(x).strip()
        if '.' in s:
            # nanosecond timestamp format — take the fractional digits as integer
            ns_str = s.split('.')[-1]   # e.g. '000000008'
            return float(int(ns_str))   # strip leading zeros -> 8.0
        return float(s)
    except (ValueError, TypeError):
        return np.nan

df_raw['delivery_time_hours'] = df_raw['delivery_time_hours'].apply(extract_hours)
df_raw['expected_time_hours'] = df_raw['expected_time_hours'].apply(extract_hours)

print('Parsed delivery_time_hours sample:', df_raw['delivery_time_hours'].dropna().head(10).tolist())
print('Parsed expected_time_hours  sample:', df_raw['expected_time_hours'].dropna().head(10).tolist())

# ── Cast remaining columns ────────────────────────────────────────────────────
for col, kind in DTYPE_MAP.items():
    if col not in df_raw.columns:
        continue
    if col in ['delivery_time_hours', 'expected_time_hours']:
        continue
    if kind == 'num':
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
    else:
        df_raw[col] = df_raw[col].astype(str).str.strip().str.lower()

# ── Drop invalid rows (zero or negative hours are physically impossible) ──────
before = len(df_raw)
df_raw = df_raw[
    (df_raw['delivery_time_hours'] > 0) &
    (df_raw['expected_time_hours'] > 0)
].copy()
print(f'\nRows after cleaning: {len(df_raw):,} (dropped {before - len(df_raw):,})')

# ── Drop ID ───────────────────────────────────────────────────────────────────
df_raw = df_raw.drop(columns=['delivery_id'], errors='ignore')

# ── Final report ──────────────────────────────────────────────────────────────
print()
print(df_raw.dtypes)
print('\nShape:', df_raw.shape)
print('\nDelayed column distribution:')
print(df_raw['delayed'].value_counts(normalize=True))
print('\nTime column stats:')
print(df_raw[['delivery_time_hours', 'expected_time_hours']].describe())

df_raw.head(5)


In [ ]:
# Schema snapshot
info = pd.DataFrame({
    'dtype'     : df_raw.dtypes,
    'null_count': df_raw.isnull().sum(),
    'null_%'    : (df_raw.isnull().sum() / len(df_raw) * 100).round(2),
    'unique'    : df_raw.nunique()
})
print(info.to_string())

---
## 4. Data Understanding (EDA)

In [ ]:
# Numerical summary
NUM_COLS = ['distance_km','package_weight_kg','delivery_time_hours',
            'expected_time_hours','delivery_rating','delivery_cost']
print("NUMERICAL SUMMARY")
display(df_raw[NUM_COLS].describe())

In [ ]:
# ── Categorical distributions ─────────────────────────────────────────────────
CAT_COLS = ['delivery_partner','package_type','vehicle_type',
            'delivery_mode','region','weather_condition','delayed','delivery_status']

fig, axes = plt.subplots(4, 2, figsize=(16, 22))
axes = axes.flatten()

for i, col in enumerate(CAT_COLS):
    vc     = df_raw[col].value_counts()
    colors = sns.color_palette('husl', len(vc))
    axes[i].bar(vc.index, vc.values, color=colors, edgecolor='white')
    axes[i].set_title(col.replace('_',' ').title(), fontweight='bold', fontsize=11)
    axes[i].tick_params(axis='x', rotation=30, labelsize=8)
    for bar, v in zip(axes[i].patches, vc.values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                     f'{v:,}', ha='center', fontsize=7)

plt.suptitle('Categorical Feature Distributions', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Numerical distributions ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    data = df_raw[col].dropna()
    axes[i].hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].axvline(data.median(), color='red', linestyle='--', linewidth=1.5,
                    label=f'Median: {data.median():.1f}')
    axes[i].set_title(col.replace('_',' ').title(), fontweight='bold')
    axes[i].legend(fontsize=8)

plt.suptitle('Numerical Feature Distributions', fontsize=14,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr = df_raw[NUM_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax, annot_kws={'size': 10})
ax.set_title('Correlation Heatmap — Numerical Features',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Delay rate by categorical feature (EDA peek) ─────────────────────────────
delay_binary = (df_raw['delayed'] == 'yes').astype(int)

group_cols = ['delivery_partner','vehicle_type','delivery_mode',
              'region','weather_condition']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(group_cols):
    rate   = delay_binary.groupby(df_raw[col]).mean().sort_values(ascending=False) * 100
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(rate)))
    axes[i].bar(rate.index, rate.values, color=colors, edgecolor='white')
    axes[i].axhline(delay_binary.mean() * 100, color='navy',
                    linestyle='--', linewidth=1.5, label='Overall avg')
    axes[i].set_title(f'Delay Rate by {col.replace("_"," ").title()}',
                      fontweight='bold')
    axes[i].set_ylabel('Delay Rate (%)')
    axes[i].tick_params(axis='x', rotation=25, labelsize=8)
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)
plt.suptitle('Delay Rate by Operational Features (EDA)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 5. Data Cleaning

In [ ]:
df = df_raw.copy()
print(f"Starting shape: {df.shape}")

# ── 5.1 Remove exact duplicates ───────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates()
print(f"[1] Duplicates removed  : {before - len(df):,}  → {len(df):,} rows")

# ── 5.2 Drop rows where core numeric fields are null ─────────────────────────
core_num = ['distance_km','package_weight_kg',
            'delivery_time_hours','expected_time_hours']
before = len(df)
df = df.dropna(subset=core_num)
print(f"[2] Null core-numeric   : {before - len(df):,} rows dropped")

# ── 5.3 Sanity-check numeric ranges ──────────────────────────────────────────
# Negative distances/weights/hours are physically impossible
for col in core_num:
    n_neg = (df[col] < 0).sum()
    if n_neg:
        df.loc[df[col] < 0, col] = np.nan
        df[col].fillna(df[col].median(), inplace=True)
        print(f"[3] {col}: {n_neg} negative values → replaced with median")

# ── 5.4 Impute remaining nulls ────────────────────────────────────────────────
def report_nulls(d):
    nulls = d.isnull().sum()
    nulls = nulls[nulls > 0].sort_values(ascending=False)
    if len(nulls) == 0:
        print("  → No missing values")
    for col, cnt in nulls.items():
        print(f"  {col}: {cnt:,} ({cnt/len(d)*100:.1f}%)")

print("\nMissing values BEFORE imputation:")
report_nulls(df)

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].median(), inplace=True)

for col in df.select_dtypes(include='object').columns:
    if df[col].isnull().any():
        df[col].fillna(df[col].mode()[0], inplace=True)

print("\nMissing values AFTER imputation:")
report_nulls(df)

print(f"\n✅ Cleaning complete. Final shape: {df.shape}")

---
## 6. Target Variable Creation

In [ ]:
# ── 6.1 Classification target: delay_flag ─────────────────────────────────────
# SINGLE SOURCE OF TRUTH: use the ground-truth 'delayed' column only.
# The earlier duplicate from Section 3 has been removed.
df['delay_flag'] = (df['delayed'] == 'yes').astype(int)

# ── 6.2 Regression target: delay_duration ────────────────────────────────────
# Positive = arrived late; negative = arrived early
# Intentional: model predicts both directions (kept as requested).
df['delay_duration'] = df['delivery_time_hours'] - df['expected_time_hours']

# ── 6.3 Validate consistency between ground truth and computed flag ───────────
flag_from_dur = (df['delay_duration'] > 0).astype(int)
agreement     = (df['delay_flag'] == flag_from_dur).mean()
print(f'Ground-truth vs computed (duration>0) agreement: {agreement*100:.1f}%')
if agreement < 0.95:
    print('WARNING: Agreement < 95% — using ground-truth label as authoritative target.')
else:
    print('OK: Ground-truth label and computed flag are highly consistent.')

vc = df['delay_flag'].value_counts()
print(f'\nClass balance:')
print(f'  On-time  (0): {vc.get(0, 0):,}  ({vc.get(0,0)/len(df)*100:.1f}%)')
print(f'  Delayed  (1): {vc.get(1, 0):,}  ({vc.get(1,0)/len(df)*100:.1f}%)')

delayed_dur = df.loc[df['delay_flag']==1, 'delay_duration']
print(f'\nDelay duration (delayed only):')
print(delayed_dur.describe())


In [ ]:
# ── 6.4 Visualise targets ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class balance
vals   = df['delay_flag'].value_counts().sort_index().values
labels = ['On-time (0)', 'Delayed (1)']
bars   = axes[0].bar(labels, vals, color=['#2ecc71','#e74c3c'], edgecolor='white')
for bar, v in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 80,
                 f'{v:,}\n({v/sum(vals)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_title('Class Balance', fontweight='bold')
axes[0].set_ylabel('Count')

# Delay duration distribution
axes[1].hist(delayed_dur, bins=50, color='#e74c3c', edgecolor='white', alpha=0.85)
axes[1].axvline(delayed_dur.median(), color='navy', linestyle='--',
                label=f'Median = {delayed_dur.median():.2f}h')
axes[1].set_title('Delay Duration Distribution (Delayed Only)', fontweight='bold')
axes[1].set_xlabel('Delay (hours)')
axes[1].legend()

# Box plot
df.boxplot(column='delay_duration', by='delay_flag', ax=axes[2],
           patch_artist=True)
axes[2].set_title('Delay Duration by Class', fontweight='bold')
axes[2].set_xlabel('delay_flag  (0=On-time, 1=Delayed)')
axes[2].set_ylabel('Duration (hours)')
plt.suptitle('Target Variable Overview', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Feature Engineering

In [ ]:
# ── 7.1 Delivery mode speed tier ─────────────────────────────────────────────
# Encode the inherent urgency / SLA tightness of each delivery mode
speed_map = {'same day': 3, 'express': 2, 'two day': 1}
df['mode_speed_tier'] = df['delivery_mode'].map(speed_map).fillna(1).astype(int)
print("mode_speed_tier:", df['mode_speed_tier'].value_counts().to_dict())

In [ ]:
# ── 7.2 Distance buckets ─────────────────────────────────────────────────────
df['distance_bucket'] = pd.qcut(
    df['distance_km'], q=4,
    labels=['short','medium','long','very_long'],
    duplicates='drop'
).astype(str)
print("distance_bucket:", df['distance_bucket'].value_counts().to_dict())

In [ ]:
# ── 7.3 Weight categories ─────────────────────────────────────────────────────
df['weight_category'] = pd.qcut(
    df['package_weight_kg'], q=3,
    labels=['light','medium','heavy'],
    duplicates='drop'
).astype(str)
print("weight_category:", df['weight_category'].value_counts().to_dict())

In [ ]:
# ── 7.4 Schedule tightness — required speed ───────────────────────────────────
# How fast (km/h) must the vehicle travel to meet the expected window?
# Higher value = tighter schedule = higher delay risk
df['speed_required_kmh'] = df['distance_km'] / (df['expected_time_hours'] + 1e-6)

# ── 7.5 Interaction: weight × distance load density ──────────────────────────
df['weight_distance_ratio'] = df['package_weight_kg'] / (df['distance_km'] + 1e-6)

# ── 7.6 Cost efficiency per km ────────────────────────────────────────────────
df['cost_per_km'] = df['delivery_cost'] / (df['distance_km'] + 1e-6)

print(f"speed_required_kmh  — mean: {df['speed_required_kmh'].mean():.2f}")
print(f"weight_distance_ratio — mean: {df['weight_distance_ratio'].mean():.4f}")
print(f"cost_per_km           — mean: {df['cost_per_km'].mean():.2f}")

In [ ]:
# ── 7.7  Route & Partner Encodings — placeholders (filled post-split) ────────
#
# LEAKAGE FIX: Previously computed on full df before splitting.
# Now we only create placeholder NaN columns here.
# Actual values are computed from TRAINING data only in Section 8.2.

df['route_key']          = df['region'] + '_' + df['delivery_mode']
df['route_delay_rate']   = np.nan   # filled after split — see Section 8.2
df['route_volume']       = np.nan
df['partner_delay_rate'] = np.nan
df['weather_delay_rate'] = np.nan

print('✅ Encoding columns reserved (NaN placeholders).')
print('   Values computed from training data only in Section 8.2.')
print(f'   Unique route keys : {df["route_key"].nunique()}')
print(f'   Unique partners   : {df["delivery_partner"].nunique()}')
print(f'   Unique weather    : {df["weather_condition"].nunique()}')


In [ ]:
# ── 7.10 Binary flags ─────────────────────────────────────────────────────────
# Is this a premium / high-cost shipment?
cost_p75 = df['delivery_cost'].quantile(0.75)
df['is_high_value'] = (df['delivery_cost'] >= cost_p75).astype(int)

# Heavy package on a very long route — highest operational risk
df['is_heavy_longhaul'] = (
    (df['weight_category'] == 'heavy') &
    (df['distance_bucket'] == 'very_long')
).astype(int)

print(f"is_high_value     : {df['is_high_value'].sum():,} shipments")
print(f"is_heavy_longhaul : {df['is_heavy_longhaul'].sum():,} shipments")
print(f"\n✅ Feature engineering done.  Total columns: {df.shape[1]}")

In [ ]:
# ── 7.11 Synthetic Dispatch Date Features ────────────────────────────────────
#
# The dataset has no real date column — the time columns encode hours only.
# We generate a realistic dispatch_date for each row by:
#   1. Seeding a deterministic offset (days) from delivery_cost + distance_km
#      so the same row always gets the same date (reproducible).
#   2. Spreading dates across one calendar year (2023-01-01 → 2023-12-31).
#
# This lets us engineer seasonality, weekday, and quarter features that have
# genuine predictive signal in real logistics data.

import hashlib

BASE_DATE = pd.Timestamp('2023-01-01')
YEAR_DAYS = 365

# Deterministic day offset: hash of cost + distance gives a stable pseudo-random int
def _day_offset(row):
    key = f"{row['delivery_cost']:.2f}_{row['distance_km']:.2f}_{row['delivery_partner']}"
    h   = int(hashlib.md5(key.encode()).hexdigest(), 16)
    return h % YEAR_DAYS

df['dispatch_date'] = df.apply(
    lambda r: BASE_DATE + pd.Timedelta(days=_day_offset(r)), axis=1
)

# ── Basic calendar features ───────────────────────────────────────────────────
df['dispatch_month']     = df['dispatch_date'].dt.month          # 1–12
df['dispatch_dayofweek'] = df['dispatch_date'].dt.dayofweek      # 0=Mon … 6=Sun
df['dispatch_quarter']   = df['dispatch_date'].dt.quarter        # 1–4
df['dispatch_dayofyear'] = df['dispatch_date'].dt.dayofyear      # 1–365
df['is_weekend']         = (df['dispatch_dayofweek'] >= 5).astype(int)

# ── Advanced temporal features ────────────────────────────────────────────────

# Week of month (1–5) — captures start/end-of-month rush
df['dispatch_week_of_month'] = df['dispatch_date'].apply(
    lambda d: (d.day - 1) // 7 + 1
)

# Peak season flag — Nov & Dec historically have higher delay rates (holiday rush)
df['is_peak_season'] = df['dispatch_month'].isin([11, 12]).astype(int)

# Month-end pressure — last 3 days of each month (quota/cutoff effect)
df['is_month_end'] = (df['dispatch_date'].dt.day >= 28).astype(int)

# Cyclic encoding of month and day-of-week
# Raw integer encoding implies ordinal distance (Dec=12 far from Jan=1) — wrong.
# Sin/cos encoding makes the cycle continuous and distance-preserving.
df['month_sin'] = np.sin(2 * np.pi * df['dispatch_month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['dispatch_month'] / 12)
df['dow_sin']   = np.sin(2 * np.pi * df['dispatch_dayofweek'] / 7)
df['dow_cos']   = np.cos(2 * np.pi * df['dispatch_dayofweek'] / 7)

# Drop the raw datetime column — not usable by sklearn directly
df = df.drop(columns=['dispatch_date'])

# ── Sanity check ──────────────────────────────────────────────────────────────
date_feats = [
    'dispatch_month', 'dispatch_dayofweek', 'dispatch_quarter',
    'dispatch_dayofyear', 'is_weekend', 'dispatch_week_of_month',
    'is_peak_season', 'is_month_end',
    'month_sin', 'month_cos', 'dow_sin', 'dow_cos',
]
print('New date features added:', len(date_feats))
print(df[date_feats].describe().round(3).to_string())

# ── Visualise delay rate by key temporal features ─────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, col, title in zip(
    axes,
    ['dispatch_month', 'dispatch_dayofweek', 'dispatch_quarter', 'is_weekend'],
    ['Delay Rate by Month', 'Delay Rate by Day of Week',
     'Delay Rate by Quarter', 'Weekday vs Weekend']
):
    rate = df.groupby(col)['delay_flag'].mean() * 100
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(rate)))
    ax.bar(rate.index.astype(str), rate.values, color=colors, edgecolor='white')
    ax.axhline(df['delay_flag'].mean() * 100, color='navy',
               linestyle='--', linewidth=1.5, label='Overall avg')
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel('Delay Rate (%)')
    ax.legend(fontsize=8)

plt.suptitle('Delay Rate by Synthetic Dispatch Date Features',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\nTotal columns now: {df.shape[1]}')


---
## 8. Modeling

In [ ]:
# ── 8.1 Define feature sets ───────────────────────────────────────────────────
#
# LEAKAGE NOTES:
#   delivery_time_hours  → IS the actual outcome; excluded.
#   expected_time_hours  → EXCLUDED (FIX): it is part of delay_duration formula.
#                          Including it lets models reconstruct the target trivially.
#   delivery_status      → post-delivery; excluded.
#   delivery_rating      → post-delivery; excluded.
#   delivery_cost        → assumed known at booking time (dispatch-time feature).
#                          Exclude if only finalised post-delivery in your system.
#   delayed              → text version of the target; excluded.
#   delay_flag           → target (classification).
#   delay_duration       → target (regression).
#   route_key            → high-cardinality key; aggregates kept instead.

LEAKAGE_COLS = [
    'delay_flag', 'delay_duration',
    'delivery_time_hours',
    'expected_time_hours',          # FIXED: was missing — leaks target
    'delayed', 'delivery_status', 'delivery_rating',
    'delivery_id'
]

feature_cols = [c for c in df.columns if c not in LEAKAGE_COLS]
X = df[feature_cols].copy()

NUM_FEATS = X.select_dtypes(include=[np.number]).columns.tolist()
CAT_FEATS = X.select_dtypes(include=['object','category']).columns.tolist()

print(f'Total feature columns: {len(feature_cols)}')
print(f'  Numerical  ({len(NUM_FEATS)}): {NUM_FEATS}')
print(f'  Categorical({len(CAT_FEATS)}): {CAT_FEATS}')


In [ ]:
# ── 8.2 Train / test split + leak-free target encoding ──────────────────────

y_cls = df['delay_flag']
y_reg = df['delay_duration']

(
    X_train, X_test,
    y_cls_train, y_cls_test,
    y_reg_train, y_reg_test
) = train_test_split(
    X, y_cls, y_reg,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_cls
)

print(f'Train : {len(X_train):,}  |  Test : {len(X_test):,}')
print(f'Delay rate — Train: {y_cls_train.mean()*100:.1f}%  '
      f'| Test: {y_cls_test.mean()*100:.1f}%')

# ── Leak-free target encoding ─────────────────────────────────────────────────
# All aggregate stats computed ONLY on training rows, then mapped to test.

_train_df = X_train.copy()
_train_df['__target__'] = y_cls_train.values
global_delay_rate = float(y_cls_train.mean())

# Route: delay rate + volume
route_stats = (
    _train_df.groupby('route_key')['__target__']
    .agg(route_delay_rate='mean', route_volume='count')
    .reset_index()
)
for _X in [X_train, X_test]:
    _X.drop(columns=['route_delay_rate','route_volume'], errors='ignore', inplace=True)
X_train = X_train.merge(route_stats, on='route_key', how='left')
X_test  = X_test.merge(route_stats,  on='route_key', how='left')
X_train['route_delay_rate'] = X_train['route_delay_rate'].fillna(global_delay_rate)
X_test['route_delay_rate']  = X_test['route_delay_rate'].fillna(global_delay_rate)
X_train['route_volume']     = X_train['route_volume'].fillna(0)
X_test['route_volume']      = X_test['route_volume'].fillna(0)

# Partner delay rate
partner_stats = (
    _train_df.groupby('delivery_partner')['__target__']
    .mean().rename('partner_delay_rate').reset_index()
)
for _X in [X_train, X_test]:
    _X.drop(columns=['partner_delay_rate'], errors='ignore', inplace=True)
X_train = X_train.merge(partner_stats, on='delivery_partner', how='left')
X_test  = X_test.merge(partner_stats,  on='delivery_partner', how='left')
X_train['partner_delay_rate'] = X_train['partner_delay_rate'].fillna(global_delay_rate)
X_test['partner_delay_rate']  = X_test['partner_delay_rate'].fillna(global_delay_rate)

# Weather delay rate
weather_stats = (
    _train_df.groupby('weather_condition')['__target__']
    .mean().rename('weather_delay_rate').reset_index()
)
for _X in [X_train, X_test]:
    _X.drop(columns=['weather_delay_rate'], errors='ignore', inplace=True)
X_train = X_train.merge(weather_stats, on='weather_condition', how='left')
X_test  = X_test.merge(weather_stats,  on='weather_condition', how='left')
X_train['weather_delay_rate'] = X_train['weather_delay_rate'].fillna(global_delay_rate)
X_test['weather_delay_rate']  = X_test['weather_delay_rate'].fillna(global_delay_rate)

# Rebuild feature lists now that encoding columns are populated
NUM_FEATS = X_train.select_dtypes(include=[np.number]).columns.tolist()
CAT_FEATS = X_train.select_dtypes(include=['object','category']).columns.tolist()

print(f'\nLeak-free target encoding applied (train stats only).')
print(f'  route_delay_rate   train mean: {X_train["route_delay_rate"].mean():.4f}')
print(f'  partner_delay_rate train mean: {X_train["partner_delay_rate"].mean():.4f}')
print(f'  weather_delay_rate train mean: {X_train["weather_delay_rate"].mean():.4f}')
print(f'\nFinal NUM_FEATS ({len(NUM_FEATS)}): {NUM_FEATS}')
print(f'Final CAT_FEATS ({len(CAT_FEATS)}): {CAT_FEATS}')


In [ ]:
# ── 8.3 Preprocessing pipeline builder ───────────────────────────────────────
def build_preprocessor(num_feats, cat_feats):
    """Return a fresh ColumnTransformer (median+scale for num, mode+OHE for cat)."""
    num_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler())
    ])
    cat_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    return ColumnTransformer([
        ('num', num_pipe, num_feats),
        ('cat', cat_pipe, cat_feats)
    ])

preprocessor = build_preprocessor(NUM_FEATS, CAT_FEATS)

print("✅ Preprocessor ready.")

In [ ]:
# ── 8.4 Generic training helpers ──────────────────────────────────────────────

def train_clf(name, clf, Xtr, ytr, Xte, yte):
    pipe = Pipeline([('pre', build_preprocessor(NUM_FEATS, CAT_FEATS)), ('clf', clf)])
    pipe.fit(Xtr, ytr)
    yp   = pipe.predict(Xte)
    yprb = pipe.predict_proba(Xte)[:, 1] if hasattr(clf, 'predict_proba') else None
    m = dict(
        name=name, pipeline=pipe, y_pred=yp, y_proba=yprb,
        accuracy  = accuracy_score(yte, yp),
        precision = precision_score(yte, yp, zero_division=0),
        recall    = recall_score(yte, yp, zero_division=0),
        f1        = f1_score(yte, yp, zero_division=0),
        roc_auc   = roc_auc_score(yte, yprb) if yprb is not None else np.nan
    )
    print(f"  [{name:<28}]  F1={m['f1']:.4f}  "
          f"AUC={m['roc_auc']:.4f}  Acc={m['accuracy']:.4f}")
    return m


def train_reg(name, reg, Xtr, ytr, Xte, yte):
    pipe = Pipeline([('pre', build_preprocessor(NUM_FEATS, CAT_FEATS)), ('reg', reg)])
    pipe.fit(Xtr, ytr)
    yp = pipe.predict(Xte)
    m = dict(
        name=name, pipeline=pipe, y_pred=yp,
        mae  = mean_absolute_error(yte, yp),
        rmse = np.sqrt(mean_squared_error(yte, yp)),
        r2   = r2_score(yte, yp)
    )
    print(f"  [{name:<28}]  MAE={m['mae']:.4f}  "
          f"RMSE={m['rmse']:.4f}  R²={m['r2']:.4f}")
    return m

print("✅ Helper functions defined.")

In [ ]:
# ── 8.5 Cross-Validation Check (5-fold Stratified) ──────────────────────────
# Provides reliable generalisation estimates (mean +/- std) before full training.

from sklearn.model_selection import StratifiedKFold, cross_validate

cv_models = {
    'Random Forest'  : RandomForestClassifier(n_estimators=100, max_depth=10,
                                               class_weight='balanced',
                                               random_state=RANDOM_STATE, n_jobs=-1),
    'HistGradient'   : HistGradientBoostingClassifier(max_iter=200, max_depth=5,
                                                       learning_rate=0.05,
                                                       random_state=RANDOM_STATE),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('5-Fold Stratified Cross-Validation (training data)\n' + '='*55)
for cv_name, cv_mdl in cv_models.items():
    cv_pipe = Pipeline([('pre', build_preprocessor(NUM_FEATS, CAT_FEATS)), ('clf', cv_mdl)])
    cv_res  = cross_validate(cv_pipe, X_train, y_cls_train, cv=skf,
                              scoring=['f1','roc_auc'], n_jobs=-1)
    print(f'  {cv_name:<22}  '
          f'F1={cv_res["test_f1"].mean():.4f} +/- {cv_res["test_f1"].std():.4f}  '
          f'AUC={cv_res["test_roc_auc"].mean():.4f} +/- {cv_res["test_roc_auc"].std():.4f}')
print()

# ── 8.6 Train all classifiers ─────────────────────────────────────────────────
print('Training classifiers on full training set...\n')

scale_pos = (y_cls_train == 0).sum() / max((y_cls_train == 1).sum(), 1)

clfs_to_train = [
    ('Logistic Regression',
     LogisticRegression(max_iter=1000, class_weight='balanced',
                        random_state=RANDOM_STATE)),
    ('Random Forest',
     RandomForestClassifier(n_estimators=300, max_depth=12,
                             class_weight='balanced',
                             random_state=RANDOM_STATE, n_jobs=-1)),
    ('Extra Trees',
     ExtraTreesClassifier(n_estimators=300, max_depth=12,
                           class_weight='balanced',
                           random_state=RANDOM_STATE, n_jobs=-1)),
    ('Gradient Boosting',
     GradientBoostingClassifier(n_estimators=300, max_depth=5,
                                 learning_rate=0.05, subsample=0.8,
                                 random_state=RANDOM_STATE)),
    ('HistGradient Boosting',
     HistGradientBoostingClassifier(max_iter=400, max_depth=6,
                                     learning_rate=0.05,
                                     random_state=RANDOM_STATE)),
]
if XGB_AVAILABLE:
    clfs_to_train.append((
        'XGBoost',
        xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                           scale_pos_weight=scale_pos, eval_metric='logloss',
                           random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    ))
if LGB_AVAILABLE:
    clfs_to_train.append((
        'LightGBM',
        lgb.LGBMClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                            is_unbalance=True, random_state=RANDOM_STATE,
                            n_jobs=-1, verbose=-1)
    ))
if CB_AVAILABLE:
    clfs_to_train.append((
        'CatBoost',
        CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05,
                            auto_class_weights='Balanced',
                            random_seed=RANDOM_STATE, verbose=0)
    ))

clf_results = []
for name, model in clfs_to_train:
    clf_results.append(train_clf(name, model,
                                  X_train, y_cls_train, X_test, y_cls_test))

clf_results.sort(key=lambda x: x['f1'], reverse=True)
best_clf = clf_results[0]
print(f'\nBest Classifier: {best_clf["name"]}  (F1={best_clf["f1"]:.4f})')


In [ ]:
# ── 8.6 Train all regressors — ALL shipments (positive=late, negative=early) ──
print("Training regressors on ALL shipments…\n")
print("  Positive delay_duration = arrived LATE")
print("  Negative delay_duration = arrived EARLY\n")

# ✅ Use ALL data — not just delayed — so model predicts both late & early
Xr_tr, yr_tr = X_train, y_reg_train
Xr_te, yr_te = X_test,  y_reg_test

print(f"Regression train: {len(Xr_tr):,}  |  test: {len(Xr_te):,}")
print(f"Duration range  : {yr_tr.min():.2f}h  →  {yr_tr.max():.2f}h")

regs_to_train = [
    ('Linear Regression',
     LinearRegression()),
    ('Ridge Regression',
     Ridge(alpha=10.0)),
    ('Lasso Regression',
     Lasso(alpha=0.1, max_iter=2000)),
    ('ElasticNet',
     ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=2000)),
    ('Random Forest Regressor',
     RandomForestRegressor(n_estimators=300, max_depth=12,
                            random_state=RANDOM_STATE, n_jobs=-1)),
    ('Extra Trees Regressor',
     ExtraTreesRegressor(n_estimators=300, max_depth=12,
                          random_state=RANDOM_STATE, n_jobs=-1)),
    ('Gradient Boosting Reg.',
     GradientBoostingRegressor(n_estimators=300, max_depth=5,
                                learning_rate=0.05, subsample=0.8,
                                random_state=RANDOM_STATE)),
    ('HistGradient Boosting Reg.',
     HistGradientBoostingRegressor(max_iter=400, max_depth=6,
                                    learning_rate=0.05,
                                    random_state=RANDOM_STATE)),
]
if XGB_AVAILABLE:
    regs_to_train.append((
        'XGBoost Regressor',
        xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                          random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    ))
if LGB_AVAILABLE:
    regs_to_train.append((
        'LightGBM Regressor',
        lgb.LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                           random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    ))
if CB_AVAILABLE:
    regs_to_train.append((
        'CatBoost Regressor',
        CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05,
                           random_seed=RANDOM_STATE, verbose=0)
    ))

reg_results = []
for name, model in regs_to_train:
    reg_results.append(train_reg(name, model, Xr_tr, yr_tr, Xr_te, yr_te))

reg_results.sort(key=lambda x: x['rmse'])
best_reg = reg_results[0]
print(f"\n🏆 Best Regressor: {best_reg['name']}  "
      f"(RMSE={best_reg['rmse']:.4f}  R²={best_reg['r2']:.4f})")


---
## 9. Evaluation

In [ ]:
# ── 9.1 Full classification report ───────────────────────────────────────────
print(f"{'='*60}")
print(f"CLASSIFICATION REPORT — {best_clf['name']}")
print(f"{'='*60}")
print(classification_report(y_cls_test, best_clf['y_pred'],
                             target_names=['On-time','Delayed']))

clf_lb = pd.DataFrame([{
    'Model'    : r['name'],
    'Accuracy' : r['accuracy'],
    'Precision': r['precision'],
    'Recall'   : r['recall'],
    'F1'       : r['f1'],
    'ROC-AUC'  : r['roc_auc']
} for r in clf_results])
print("\nCLASSIFICATION LEADERBOARD")
print(clf_lb.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── 9.2 Classification visuals ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion matrix
cm = confusion_matrix(y_cls_test, best_clf['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['On-time','Delayed'],
            yticklabels=['On-time','Delayed'], linewidths=0.5)
axes[0].set_title(f'Confusion Matrix\n{best_clf["name"]}', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')

# ROC curves
for r in clf_results:
    if r['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_cls_test, r['y_proba'])
        axes[1].plot(fpr, tpr, linewidth=1.8,
                     label=f"{r['name']} ({r['roc_auc']:.3f})")
axes[1].plot([0,1],[0,1],'k--', linewidth=1)
axes[1].set_title('ROC Curves — All Classifiers', fontweight='bold')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=7)

# F1 vs AUC bar chart
names = [r['name'] for r in clf_results]
x     = np.arange(len(names))
axes[2].bar(x - 0.2, [r['f1']      for r in clf_results], 0.35,
            label='F1',      color='steelblue',  alpha=0.85)
axes[2].bar(x + 0.2, [r['roc_auc'] for r in clf_results], 0.35,
            label='ROC-AUC', color='darkorange', alpha=0.85)
axes[2].set_xticks(x)
axes[2].set_xticklabels(names, rotation=20, ha='right', fontsize=8)
axes[2].set_ylim(0, 1.15)
axes[2].set_title('F1 vs ROC-AUC by Model', fontweight='bold')
axes[2].legend()

plt.suptitle('Classification Evaluation', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 9.3 Regression leaderboard ────────────────────────────────────────────────
reg_lb = pd.DataFrame([{
    'Model': r['name'], 'MAE': r['mae'], 'RMSE': r['rmse'], 'R²': r['r2']
} for r in reg_results]).sort_values('RMSE')
print("REGRESSION LEADERBOARD")
print("="*55)
print(reg_lb.to_string(index=False, float_format='{:.4f}'.format))

In [ ]:
# ── 9.4 Regression visuals ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

yp_r = best_reg['y_pred']

# Actual vs Predicted
axes[0].scatter(yr_te, yp_r, alpha=0.35, color='steelblue',
                edgecolor='none', s=18)
lims = [min(yr_te.min(), yp_r.min()), max(yr_te.max(), yp_r.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
axes[0].set_title(f'Actual vs Predicted\n{best_reg["name"]}', fontweight='bold')
axes[0].set_xlabel('Actual Delay (h)')
axes[0].set_ylabel('Predicted Delay (h)')
axes[0].legend()

# Residuals
residuals = yr_te - yp_r
axes[1].hist(residuals, bins=50, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='navy', linestyle='--')
axes[1].set_title('Residual Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (h)')

# Regressor comparison
rnames = reg_lb['Model'].tolist()
rx     = np.arange(len(rnames))
axes[2].bar(rx, reg_lb['RMSE'].tolist(), color='steelblue', alpha=0.85, label='RMSE')
ax2 = axes[2].twinx()
ax2.plot(rx, reg_lb['R²'].tolist(), 'D-', color='darkorange',
         linewidth=2, markersize=7, label='R²')
axes[2].set_xticks(rx)
axes[2].set_xticklabels(rnames, rotation=20, ha='right', fontsize=7)
axes[2].set_title('Regressor Comparison', fontweight='bold')
axes[2].set_ylabel('RMSE'); ax2.set_ylabel('R²')
axes[2].legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)

plt.suptitle('Regression Evaluation', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## 10. Explainability (SHAP)

In [ ]:
# ── 10.1 Extract transformed feature names ────────────────────────────────────
def get_feature_names(prep, num_feats, cat_feats):
    ohe_names = (
        prep.named_transformers_['cat']['ohe']
        .get_feature_names_out(cat_feats)
        .tolist()
    )
    return num_feats + ohe_names

clf_pre  = best_clf['pipeline'].named_steps['pre']
clf_mdl  = best_clf['pipeline'].named_steps['clf']

feat_names  = get_feature_names(clf_pre, NUM_FEATS, CAT_FEATS)
X_test_t    = pd.DataFrame(clf_pre.transform(X_test), columns=feat_names)

print(f"Transformed test shape  : {X_test_t.shape}")
print(f"First 8 feature names   : {feat_names[:8]}")

In [ ]:
X_check = best_clf['pipeline'][:-1].transform(X_train)
print("Features after preprocessing:", X_check.shape[1])
clf_step = best_clf['pipeline'].named_steps['clf']
if hasattr(clf_step, 'coef_'):
    print("Model expects (coef_):", clf_step.coef_.shape[1])
elif hasattr(clf_step, 'n_features_in_'):
    print("Model n_features_in_:", clf_step.n_features_in_)
else:
    print("Model type:", type(clf_step).__name__, "— feature count matches preprocessor output ✅")


In [ ]:
# ── 10.2 SHAP — TreeExplainer (fast+accurate) with KernelExplainer fallback ──
if SHAP_AVAILABLE:

    clf_pre_shap = best_clf['pipeline'].named_steps['pre']
    clf_mdl_shap = best_clf['pipeline'].named_steps['clf']

    feat_names_shap = get_feature_names(clf_pre_shap, NUM_FEATS, CAT_FEATS)
    X_test_t  = pd.DataFrame(clf_pre_shap.transform(X_test),  columns=feat_names_shap)
    X_train_t = pd.DataFrame(clf_pre_shap.transform(X_train), columns=feat_names_shap)

    # Use TreeExplainer for tree models (100-1000x faster, exact SHAP)
    _tree_types = [RandomForestClassifier, ExtraTreesClassifier,
                   GradientBoostingClassifier, HistGradientBoostingClassifier]
    if XGB_AVAILABLE: _tree_types.append(xgb.XGBClassifier)
    if LGB_AVAILABLE: _tree_types.append(lgb.LGBMClassifier)
    if CB_AVAILABLE:  _tree_types.append(CatBoostClassifier)

    if isinstance(clf_mdl_shap, tuple(_tree_types)):
        explainer_type = 'TreeExplainer'
        explainer = shap.TreeExplainer(clf_mdl_shap)
        sample_data_t = X_test_t.sample(min(500, len(X_test_t)), random_state=42)
        sv_raw = explainer.shap_values(sample_data_t)
    else:
        explainer_type = 'KernelExplainer (fallback — linear model)'
        bg = shap.sample(X_train_t, min(500, len(X_train_t)), random_state=42)
        explainer = shap.KernelExplainer(clf_mdl_shap.predict_proba, bg)
        sample_data_t = X_test_t.sample(min(500, len(X_test_t)), random_state=42)
        sv_raw = explainer.shap_values(sample_data_t)

    # Normalise to 2D array for class 1
    if isinstance(sv_raw, list):
        sv = sv_raw[1]
    elif isinstance(sv_raw, np.ndarray) and sv_raw.ndim == 3:
        sv = sv_raw[:, :, 1]
    else:
        sv = sv_raw

    sv_feat_names = feat_names_shap

    print(f'SHAP computed using: {explainer_type}')
    print(f'sv shape          : {sv.shape}')
    print(f'sample_data shape : {sample_data_t.shape}')


In [ ]:
# ── 10.3 Global feature importance ────────────────────────────────────────────
if SHAP_AVAILABLE:
    mean_abs = np.abs(sv).mean(axis=0)
    shap_imp = (
        pd.Series(mean_abs, index=sv_feat_names)
        .sort_values(ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(10, 7))
    colors  = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(shap_imp)))
    shap_imp.sort_values().plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'Global SHAP Feature Importance (Top 20)\n{best_clf["name"]}',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Mean |SHAP Value|')
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 10.4 Beeswarm (summary) plot ─────────────────────────────────────────────
if SHAP_AVAILABLE:
    top20_names = shap_imp.index.tolist()
    t_idx       = [sv_feat_names.index(f) for f in top20_names if f in sv_feat_names]

    plt.figure(figsize=(10, 8))
    shap.summary_plot(
        sv[:, t_idx],
        sample_data_t.iloc[:, t_idx],
        feature_names=[sv_feat_names[i] for i in t_idx],
        show=False, plot_size=None
    )
    plt.title(f'SHAP Beeswarm Plot — {best_clf["name"]}',
              fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.show()


In [ ]:
# ── 10.5 Local explanation — one delayed shipment ─────────────────────────────
if SHAP_AVAILABLE:
    delayed_idx = np.where(y_cls_test.values == 1)[0]
    sample_i    = delayed_idx[0]

    print(f'Sample index : {sample_i}')
    print(f'True label   : {"Delayed" if y_cls_test.iloc[sample_i]==1 else "On-time"}')
    print(f'Predicted    : {"Delayed" if best_clf["y_pred"][sample_i]==1 else "On-time"}')
    print(f'Delay prob   : {best_clf["y_proba"][sample_i]:.3f}')

    # Transform the single sample into preprocessed space
    clf_pre_local = best_clf['pipeline'].named_steps['pre']
    single_row    = X_test.iloc[[sample_i]]
    single_t      = pd.DataFrame(clf_pre_local.transform(single_row), columns=sv_feat_names)

    # Compute SHAP for this single instance
    sv_single_raw = explainer.shap_values(single_t)
    if isinstance(sv_single_raw, list):
        sv_s = sv_single_raw[1][0]
    elif isinstance(sv_single_raw, np.ndarray) and sv_single_raw.ndim == 3:
        sv_s = sv_single_raw[0, :, 1]
    else:
        sv_s = sv_single_raw[0]

    N     = 15
    top_i = np.argsort(np.abs(sv_s))[::-1][:N]

    lf  = [sv_feat_names[i] for i in top_i]
    lv  = sv_s[top_i]
    lfv = single_t.iloc[0, top_i].values

    fig, ax = plt.subplots(figsize=(10, 6))
    colors_l = ['#e74c3c' if v > 0 else '#2ecc71' for v in lv[::-1]]
    y_pos    = np.arange(N)
    ax.barh(y_pos, lv[::-1], color=colors_l, edgecolor='white')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(
        [f"{lf[N-1-k]} = {lfv[N-1-k] if isinstance(lfv[N-1-k], str) else f'{float(lfv[N-1-k]):.2f}'}"
         for k in range(N)],
        fontsize=9
    )
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP Value')
    ax.set_title(
        f'Local SHAP Explanation — Sample #{sample_i}\n'
        'Red = raises delay risk  |  Green = lowers delay risk',
        fontweight='bold', fontsize=11
    )
    plt.tight_layout()
    plt.show()


---
## 11. Production Pipeline & Prediction API

In [ ]:
# ── 11.1 Rebuild clean production pipelines ───────────────────────────────────
def _best_clf_model():
    n = best_clf['name']
    if 'CatBoost'  in n and CB_AVAILABLE:
        return CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05,
                                   auto_class_weights='Balanced',
                                   random_seed=RANDOM_STATE, verbose=0)
    if 'XGBoost'   in n and XGB_AVAILABLE:
        return xgb.XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                                   scale_pos_weight=scale_pos, eval_metric='logloss',
                                   random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    if 'LightGBM'  in n and LGB_AVAILABLE:
        return lgb.LGBMClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                                    is_unbalance=True, random_state=RANDOM_STATE,
                                    n_jobs=-1, verbose=-1)
    if 'HistGradient' in n:
        return HistGradientBoostingClassifier(max_iter=400, max_depth=6,
                                               learning_rate=0.05,
                                               random_state=RANDOM_STATE)
    if 'Extra Trees' in n:
        return ExtraTreesClassifier(n_estimators=300, max_depth=12,
                                     class_weight='balanced',
                                     random_state=RANDOM_STATE, n_jobs=-1)
    if 'Gradient'  in n:
        return GradientBoostingClassifier(n_estimators=300, max_depth=5,
                                           learning_rate=0.05, random_state=RANDOM_STATE)
    if 'Random Forest' in n:
        return RandomForestClassifier(n_estimators=300, max_depth=12,
                                       class_weight='balanced',
                                       random_state=RANDOM_STATE, n_jobs=-1)
    return LogisticRegression(max_iter=1000, class_weight='balanced',
                               random_state=RANDOM_STATE)

def _best_reg_model():
    n = best_reg['name']
    if 'CatBoost'  in n and CB_AVAILABLE:
        return CatBoostRegressor(iterations=400, depth=6, learning_rate=0.05,
                                  random_seed=RANDOM_STATE, verbose=0)
    if 'XGBoost'   in n and XGB_AVAILABLE:
        return xgb.XGBRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                 random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    if 'LightGBM'  in n and LGB_AVAILABLE:
        return lgb.LGBMRegressor(n_estimators=400, max_depth=6, learning_rate=0.05,
                                  random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    if 'HistGradient' in n:
        return HistGradientBoostingRegressor(max_iter=400, max_depth=6,
                                              learning_rate=0.05,
                                              random_state=RANDOM_STATE)
    if 'Extra Trees' in n:
        return ExtraTreesRegressor(n_estimators=300, max_depth=12,
                                    random_state=RANDOM_STATE, n_jobs=-1)
    if 'Gradient'  in n:
        return GradientBoostingRegressor(n_estimators=300, max_depth=5,
                                          learning_rate=0.05, random_state=RANDOM_STATE)
    if 'Random Forest' in n:
        return RandomForestRegressor(n_estimators=300, max_depth=12,
                                      random_state=RANDOM_STATE, n_jobs=-1)
    if 'Lasso' in n:
        return Lasso(alpha=0.1, max_iter=2000)
    if 'ElasticNet' in n:
        return ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=2000)
    return Ridge(alpha=10.0)

prod_clf_pipe = Pipeline([
    ('pre', build_preprocessor(NUM_FEATS, CAT_FEATS)),
    ('clf', _best_clf_model())
])
prod_reg_pipe = Pipeline([
    ('pre', build_preprocessor(NUM_FEATS, CAT_FEATS)),
    ('reg', _best_reg_model())
])

prod_clf_pipe.fit(X_train, y_cls_train)
prod_reg_pipe.fit(Xr_tr, yr_tr)   # Xr_tr = X_train (all shipments)

print(f"✅ Production pipelines fitted.")
print(f"   Classifier : {best_clf['name']}")
print(f"   Regressor  : {best_reg['name']}")


In [ ]:
if SHAP_AVAILABLE:
    prod_pre        = prod_clf_pipe.named_steps['pre']
    prod_mdl        = prod_clf_pipe.named_steps['clf']
    prod_feat_names = get_feature_names(prod_pre, NUM_FEATS, CAT_FEATS)
    X_train_t = pd.DataFrame(prod_pre.transform(X_train), columns=prod_feat_names)
    bg        = shap.sample(X_train_t, min(200, len(X_train_t)),
                            random_state=RANDOM_STATE)

    # ── All tree/ensemble models → TreeExplainer ──────────────────────────────
    tree_types = [
        RandomForestClassifier,
        GradientBoostingClassifier,
        ExtraTreesClassifier,               # was missing
        HistGradientBoostingClassifier,     # was missing
    ]
    if XGB_AVAILABLE:
        tree_types.append(xgb.XGBClassifier)
    if LGB_AVAILABLE:
        tree_types.append(lgb.LGBMClassifier)
    if CB_AVAILABLE:
        tree_types.append(CatBoostClassifier)   # was missing

    tree_types = tuple(tree_types)

    # ── Linear models → LinearExplainer ──────────────────────────────────────
    linear_types = (LogisticRegression,)        # add Ridge, SVC(kernel='linear') etc. if used

    if isinstance(prod_mdl, tree_types):
        prod_explainer = shap.TreeExplainer(prod_mdl)
    elif isinstance(prod_mdl, linear_types):
        prod_explainer = shap.LinearExplainer(prod_mdl, bg)
    else:
        # KernelExplainer works with any predict function — slower but safe
        predict_fn     = lambda x: prod_mdl.predict_proba(x)[:, 1]
        prod_explainer = shap.KernelExplainer(predict_fn, bg)

    print(f"✅ Production SHAP explainer ready  ({type(prod_explainer).__name__}).")
else:
    prod_explainer  = None
    prod_feat_names = feat_names

In [ ]:
# ── 11.3 predict_shipment_delay ───────────────────────────────────────────────

def predict_shipment_delay(
    input_data      : dict,
    clf_pipe        = prod_clf_pipe,
    reg_pipe        = prod_reg_pipe,
    explainer       = None,
    feat_names_out  = None,
    top_n           : int = 5
) -> dict:
    """
    Predict delay risk for a single shipment.

    Returns
    -------
    dict:
        delay_probability    : float  [0, 1]
        will_be_delayed      : bool
        duration_hours       : float  positive=late, negative=early arrival
        arrival_status       : str    e.g. '2.50h late' or '1.20h early'
        risk_level           : str    'Low 🟢' / 'Medium 🟡' / 'High 🔴'
        top_delay_reasons    : list of dicts (SHAP-based drivers)
    """
    row = pd.DataFrame([input_data])

    # 1 — Classification
    prob    = float(clf_pipe.predict_proba(row)[0, 1])
    delayed = bool(clf_pipe.predict(row)[0])

    risk = ('Low 🟢'    if prob < 0.35 else
            'Medium 🟡' if prob < 0.65 else
            'High 🔴')

    # 2 — Regression: predict for ALL (positive=late, negative=early)
    duration_h = float(reg_pipe.predict(row)[0])
    if duration_h > 0:
        arrival_status = f'+{duration_h:.2f}h late ⚠️'
    else:
        arrival_status = f'{abs(duration_h):.2f}h early ✅'

    # 3 — SHAP explanation
    reasons = []
    if explainer is not None and feat_names_out:
        try:
            pre    = clf_pipe.named_steps['pre']
            Xt     = pd.DataFrame(pre.transform(row), columns=feat_names_out)
            sv_loc = explainer.shap_values(Xt)
            if isinstance(sv_loc, list):
                sv_loc = sv_loc[1][0]
            elif sv_loc.ndim == 3:
                sv_loc = sv_loc[0, :, 1]
            else:
                sv_loc = sv_loc[0]
            top_i  = np.argsort(np.abs(sv_loc))[::-1][:top_n]
            reasons = [
                {
                    'rank'     : k + 1,
                    'feature'  : feat_names_out[i],
                    'value'    : round(float(Xt.iloc[0, i]), 4),
                    'shap'     : round(float(sv_loc[i]), 4),
                    'direction': ('⬆ Increases delay risk'
                                  if sv_loc[i] > 0 else
                                  '⬇ Reduces delay risk')
                }
                for k, i in enumerate(top_i)
            ]
        except Exception as e:
            reasons = [{'error': str(e)}]

    return {
        'delay_probability'  : round(prob, 4),
        'will_be_delayed'    : delayed,
        'duration_hours'     : round(duration_h, 2),
        'arrival_status'     : arrival_status,
        'risk_level'         : risk,
        'top_delay_reasons'  : reasons
    }

print("✅ predict_shipment_delay() defined.")


In [ ]:
# ── 11.4 Live demo ────────────────────────────────────────────────────────────
sample_input = X_test.iloc[5].to_dict()

result = predict_shipment_delay(
    input_data     = sample_input,
    clf_pipe       = prod_clf_pipe,
    reg_pipe       = prod_reg_pipe,
    explainer      = prod_explainer if SHAP_AVAILABLE else None,
    feat_names_out = prod_feat_names if SHAP_AVAILABLE else None
)

print("\n" + "="*62)
print("  🚚  SHIPMENT DELAY PREDICTION — LIVE INFERENCE")
print("="*62)
print(f"  Delay Probability     : {result['delay_probability']*100:.1f}%")
print(f"  Will Be Delayed       : {'YES ⚠️' if result['will_be_delayed'] else 'NO ✅'}")
print(f"  Arrival Status        : {result['arrival_status']}")
print(f"  Duration Hours        : {result['duration_hours']:+.2f}h  (+ = late, - = early)")
print(f"  Risk Level            : {result['risk_level']}")

if result['top_delay_reasons']:
    print(f"\n  Top {len(result['top_delay_reasons'])} Drivers (SHAP):")
    for r in result['top_delay_reasons']:
        if 'feature' in r:
            print(f"    {r['rank']}. {r['feature']:<35} "
                  f"SHAP={r['shap']:+.4f}  {r['direction']}")

actual_label = y_cls_test.iloc[5]
actual_dur   = y_reg_test.iloc[5]
print(f"\n  Ground Truth Label    : {'Delayed' if actual_label else 'On-time'}")
print(f"  Ground Truth Duration : {actual_dur:+.2f}h")
print("="*62)


---
## 12. Additional Visualizations

In [ ]:
# ── Model-based feature importance (sklearn models) ───────────────────────────
clf_mdl_prod = prod_clf_pipe.named_steps['clf']
clf_pre_prod = prod_clf_pipe.named_steps['pre']
fi_names     = get_feature_names(clf_pre_prod, NUM_FEATS, CAT_FEATS)

if hasattr(clf_mdl_prod, 'feature_importances_'):
    fi = (
        pd.Series(clf_mdl_prod.feature_importances_, index=fi_names)
        .sort_values(ascending=False)
        .head(20)
    )
    fig, ax = plt.subplots(figsize=(10, 7))
    colors  = plt.cm.Blues_r(np.linspace(0.2, 0.85, len(fi)))
    fi.sort_values().plot(kind='barh', ax=ax, color=colors)
    ax.set_title(f'Top 20 Feature Importances\n{best_clf["name"]}',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Importance (Gini / Split gain)')
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_clf['name']} does not expose feature_importances_.")

In [ ]:
# ── Delay rate heatmap: region × weather ──────────────────────────────────────
pivot = (
    df.groupby(['region','weather_condition'])['delay_flag']
    .mean()
    .unstack(fill_value=0) * 100
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Delay Rate (%)'})
ax.set_title('Delay Rate (%) by Region × Weather Condition',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Weather Condition')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

In [ ]:
# ── Delay rate by delivery partner × mode ─────────────────────────────────────
pivot2 = (
    df.groupby(['delivery_partner','delivery_mode'])['delay_flag']
    .mean()
    .unstack(fill_value=0) * 100
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(pivot2, annot=True, fmt='.1f', cmap='coolwarm',
            linewidths=0.5, ax=ax, center=pivot2.values.mean(),
            cbar_kws={'label': 'Delay Rate (%)'})
ax.set_title('Delay Rate (%) by Partner × Delivery Mode',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 13. Model Persistence

In [ ]:
import joblib

os.makedirs('models', exist_ok=True)

joblib.dump(prod_clf_pipe,   'models/clf_pipeline.pkl')
joblib.dump(prod_reg_pipe,   'models/reg_pipeline.pkl')
joblib.dump(prod_feat_names, 'models/feature_names.pkl')

if SHAP_AVAILABLE:
    joblib.dump(prod_explainer, 'models/shap_explainer.pkl')

meta = {
    'created'         : datetime.datetime.now().isoformat(),
    'best_classifier' : best_clf['name'],
    'clf_f1'          : round(best_clf['f1'], 4),
    'clf_auc'         : round(best_clf['roc_auc'], 4),
    'best_regressor'  : best_reg['name'],
    'reg_rmse'        : round(best_reg['rmse'], 4),
    'reg_r2'          : round(best_reg['r2'], 4),
    'num_features'    : NUM_FEATS,
    'cat_features'    : CAT_FEATS,
    'train_rows'      : int(len(X_train)),
    'test_rows'       : int(len(X_test))
}
with open('models/model_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("✅ Artifacts saved to ./models/")
for fp in sorted(os.listdir('models')):
    size = os.path.getsize(f'models/{fp}') / 1024
    print(f"   {fp:<30}  {size:>8.1f} KB")

---
## 14. Conclusion

### Results Summary

| Task | Best Model | Key Metrics |
|---|---|---|
| Classification (delay_flag) | *see cell output* | F1, ROC-AUC |
| Regression (delay_duration) | *see cell output* | RMSE, R² |

### Feature Engineering Highlights
| Feature | Rationale |
|---|---|
| `mode_speed_tier` | Encodes SLA tightness numerically |
| `distance_bucket` | Captures non-linear distance effects |
| `weight_category` | Heavy packages harder to handle/load |
| `speed_required_kmh` | Tight schedule = higher delay risk |
| `route_delay_rate` | Region+mode historical reliability |
| `partner_delay_rate` | Carrier-level reliability score |
| `weather_delay_rate` | Empirical weather risk from data |
| `cost_per_km` | Proxy for service tier / priority |
| `weight_distance_ratio` | Load density operational metric |
| `is_heavy_longhaul` | Highest-risk operational combination |

### Business Takeaways
- **Route risk**: certain region × mode combinations consistently over-perform or under-perform — use `route_delay_rate` to flag before dispatch.
- **Partner accountability**: `partner_delay_rate` can feed into SLA review reports and carrier scoring.
- **Weather response**: `weather_delay_rate` enables proactive communication when adverse weather is forecast.

### Next Steps
1. **Hyperparameter tuning** — Optuna Bayesian search (especially for XGBoost/LightGBM).
2. **Threshold calibration** — Adjust the 0.5 decision threshold to optimise the Precision/Recall trade-off per business cost.
3. **Time-based train/test split** — Ensure future leakage does not affect reported metrics.
4. **REST API** — Wrap `predict_shipment_delay()` in FastAPI for real-time scoring.
5. **Model monitoring** — Weekly drift detection (Evidently AI) on live predictions.
6. **Streamlit dashboard** — Interactive delay explorer for operations teams.

---
> **Dataset**: [Delivery Logistics Dataset — Kaggle](https://www.kaggle.com/datasets/ayeshaseherr/delivery-logistics-dataset/data)  
> **Stack**: scikit-learn · XGBoost · LightGBM · SHAP · pandas · matplotlib · seaborn